In [ ]:
!pip install pandas scikit-learn skl2onnx onnx onnxruntime

In [ ]:
import re
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.utils import resample
import joblib
from hybrid_model import HybridPredictor
import joblib

# Keyword matching for rule-based classification
CATEGORY_KEYWORDS = {
	"software": ["\\bword\\b", "\\bexcel\\b", "\\bpowerpoint\\b", "\\blibreoffice\\b", "\\boutlook\\b", "\\bapplication\\b", "\\bsoftware\\b", "\\bapp\\b", "\\bupdate\\b"],
	"hardware": ["\\bmouse\\b", "\\bkeyboard\\b", "\\bmonitor\\b", "\\bprinter\\b", "\\bhardware\\b", "\\bbattery\\b", "\\bhinge\\b", "\\bdisplay\\b", "\\bscreen\\b", "\\bspeaker\\b", "\\bspeakers\\b", "\\baudio\\b", "\\btouchpad\\b", "\\bfan\\b"],
	"access": ["\\baccess\\b", "\\bpermission\\b", "\\baccount\\b", "\\blogin\\b", "\\bcredential\\b"],
	"network": ["\\bwifi\\b", "\\bvpn\\b", "\\bnetwork\\b", "\\bconnection\\b", "\\blatency\\b", "\\bpacket\\b"],
	"security": ["\\bphishing\\b", "\\bmalware\\b", "\\bvirus\\b", "\\bsecurity\\b", "\\bbreach\\b", "\\bransomware\\b", "\\bauthentication\\b"]
}

PRIORITY_KEYWORDS = {
	"High": ["\\burgent\\b", "\\bcritical\\b", "\\bbreach\\b", "\\bdata loss\\b", "\\bimmediate\\b", "\\basap\\b", "\\bmajor\\b", "\\bdown\\b", "\\bcan't login\\b", "\\bcannot login\\b"],
	"Medium": ["\\bcannot\\b", "\\bfail\\b", "\\berror\\b", "\\bcorrupt\\b", "\\bcorrupted\\b", "\\bissue\\b", "\\bproblem\\b", "\\bintermittent\\b", "\\bslow\\b"],
	"Low": ["\\bsmall\\b", "\\bminor\\b", "\\bno pressure\\b", "\\bquestion\\b", "\\bhelp\\b", "\\brequest\\b"]
}

def keyword_match_regex(description, keywords_dict):
	description_lower = description.lower()
	for key, patterns in keywords_dict.items():
		for pat in patterns:
			if re.search(pat, description_lower):
				return key
	return None

# Load dataset
df = pd.read_csv("tickets.csv", on_bad_lines='skip')
df = df.dropna(subset=['description', 'priority', 'category'])
df['description'] = df['description'].astype(str)

# Prepare separate pipelines for category and priority
df_category = df[['description', 'category']].copy()
df_priority = df[['description', 'priority']].copy()

# Unsample classes low in number
# So all classes have equal representation
def balance_by_target(df_in, target_col):
	df_counts = df_in[target_col].value_counts()
	max_n = df_counts.max()
	parts = []
	for label, n in df_counts.items():
		part = df_in[df_in[target_col] == label]
		if n < max_n:
			part_up = resample(part, replace=True, n_samples=max_n, random_state=42)
			parts.append(part_up)
		else:
			parts.append(part)
	df_bal = pd.concat(parts).sample(frac=1, random_state=42).reset_index(drop=True)
	return df_bal

# Balance separately
df_cat_bal = balance_by_target(df_category, 'category')
df_prio_bal = balance_by_target(df_priority, 'priority')

print("\nDataset category distribution before balancing:")
print(df_category['category'].value_counts())
print("\nCategory after balancing:")
print(df_cat_bal['category'].value_counts())

print("\nDataset priority distribution before balancing:")
print(df_priority['priority'].value_counts())
print("\nPriority after balancing:")
print(df_prio_bal['priority'].value_counts())

# Split into train and test sets
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
	df_cat_bal['description'], df_cat_bal['category'], test_size=0.2, random_state=42, stratify=df_cat_bal['category']
)

Xp_train, Xp_test, yp_train, yp_test = train_test_split(
	df_prio_bal['description'], df_prio_bal['priority'], test_size=0.2, random_state=42, stratify=df_prio_bal['priority']
)

# TF-IDF + Logistic Regression pipelines
cat_pipeline = Pipeline([
	('tfidf', TfidfVectorizer(max_features=3000, stop_words='english', ngram_range=(1,2))),
	('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])

prio_pipeline = Pipeline([
	('tfidf', TfidfVectorizer(max_features=2000, stop_words='english', ngram_range=(1,2))),
	('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])

# Train both models
print("\nTraining category model...")
cat_pipeline.fit(Xc_train, yc_train)

print("\nTraining priority model...")
prio_pipeline.fit(Xp_train, yp_train)

# Save and download combined model
hybrid = HybridPredictor(
    cat_gs.best_estimator_,
    prio_gs.best_estimator_
)

joblib.dump(hybrid, "model.pkl")
print("Saved model!")


In [ ]:
# Test the model

from hybrid_model import HybridPredictor
import joblib

hybrid = joblib.load("model.pkl")

desc = input("Enter ticket description: ")
cat, prio = hybrid.predict(desc)

print("Predicted category:", cat)
print("Predicted priority:", prio)
